# LTX-Video 2B fence video (Kaggle free GPU)

**Before running:** In the right sidebar, set `Accelerator` to `GPU T4 x2` (or `GPU P100`) and turn `Internet` **On**. Then `Run All`.

This does NOT need the browser tab to stay open once you click **Save Version > Save & Run All (Commit)** -- it runs on Kaggle's servers in the background. Check the notebook's **Output** tab when the commit finishes.

Uses LTX-Video 2B (Lightricks) instead of Wan2.2 A14B -- a much lighter, faster model, purpose-built for quick iteration. Quality will be lower than A14B but generation should be several times faster.

**Caveat:** this graph is a best-effort reconstruction of the official ComfyUI LTX-Video first/last-frame example (LTXVImgToVideo + LTXVAddGuide + LTXVConditioning + LTXVScheduler/SamplerCustom). It has not been validated end-to-end yet in this project, unlike the Wan pipeline. Keep `SMOKE_TEST = True` for the first run to confirm the graph works before committing to a longer run.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
major, minor = (int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
if (major, minor) < (2, 7):
    print('Upgrading torch...')
    !pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('If this ran, restart the kernel and re-run from the top.')


In [ ]:
# Kaggle working is capped at 20GB (it's what gets persisted as Output).
# Clone ComfyUI and model weights into /kaggle/tmp instead -- use /kaggle/working
# only for the small final video (just like the Wan notebooks).
import os
os.makedirs('/kaggle/tmp', exist_ok=True)
%cd /kaggle/tmp
if not os.path.exists('/kaggle/tmp/ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /kaggle/tmp/ComfyUI
!pip install -q -r requirements.txt


In [ ]:
# LTX-Video is supported natively by recent ComfyUI (LTXVImgToVideo, LTXVAddGuide,
# LTXVConditioning, LTXVScheduler nodes ship in ComfyUI core -- no custom node needed).
import os, urllib.request

def download_if_needed(path, url):
    full = f'/kaggle/tmp/ComfyUI/{path}'
    os.makedirs(os.path.dirname(full), exist_ok=True)
    need = True
    if os.path.exists(full):
        try:
            req = urllib.request.Request(url, method='HEAD')
            with urllib.request.urlopen(req) as resp:
                remote_size = int(resp.headers.get('Content-Length', -1))
            local_size = os.path.getsize(full)
            need = remote_size != local_size
        except Exception:
            need = False
    if need:
        print(f'Downloading {path} ...')
        !wget -q --show-progress -O "{full}" "{url}"
    else:
        print(f'{path} already OK, skipping')

files_to_check = [
    ('models/checkpoints/ltx-video-2b-v0.9.safetensors', 'https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.safetensors'),
]
for path, url in files_to_check:
    download_if_needed(path, url)


In [ ]:
# Same two reference images used by the Wan notebooks (no manual upload needed).
import os
os.makedirs('/kaggle/tmp/ComfyUI/input', exist_ok=True)
%cd /kaggle/tmp/ComfyUI/input
!wget -q -O start_no_fence.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/start%20frame.png"
!wget -q -O end_full_fence.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/end%20frame%20with%20fence.png"
!ls -la


In [ ]:
# Start ComfyUI server as a background process (idempotent).
import subprocess, time, os, urllib.request

def is_server_up():
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        return True
    except Exception:
        return False

if not is_server_up():
    %cd /kaggle/tmp/ComfyUI
    log = open('/kaggle/tmp/comfyui.log', 'w')
    subprocess.Popen(['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_server_up():
            break
        time.sleep(2)
print('Server up:', is_server_up())


In [ ]:
import json, time, urllib.request

CKPT = "ltx-video-2b-v0.9.safetensors"

# SMOKE TEST: tiny settings to prove the graph (node names, inputs, tiled decode,
# ffmpeg conversion) actually works end-to-end before committing to the real size.
# This LTX graph is unverified in this project so far -- run smoke test first.
SMOKE_TEST = True

if SMOKE_TEST:
    WIDTH, HEIGHT, LENGTH = 256, 256, 17   # ~2.8s @ 6fps, tiny
    STEPS = 8
else:
    WIDTH, HEIGHT, LENGTH = 512, 320, 25  # ~4s at 6fps (testing)
    STEPS = 30

FPS = 6
CFG = 3.0
SEED = 42

POSITIVE_PROMPT = (
    "aerial drone shot flying forward over farmland village at sunset, camera "
    "continuously moving forward the entire time. A tall wooden fence pillar "
    "starts high up in the sky and gently descends, clearly and smoothly "
    "moving downward through the air over several seconds -- the pillar is "
    "seen traveling down through the frame in mid-air well before it reaches "
    "the ground, not simply appearing already in place. When it reaches the "
    "ground it plants into the earth and rights itself, rotating and "
    "straightening upright into a standing vertical fence post. More pillars "
    "follow the same gentle downward descent one after another, each one "
    "clearly seen moving down from high in the sky before landing and "
    "standing upright, forming a single line of posts stretching into the "
    "distance. Then, barbed wire unspools and stretches itself between the "
    "posts, hooking onto each one in sequence from near to far, wire strands "
    "pulling taut across the line of posts. magical stop-motion construction, "
    "cinematic lighting, photorealistic"
)
NEGATIVE_PROMPT = "blurry, low quality, distorted, flickering, artifacts, watermark, text, static camera, jerky motion, falling, dropping, crashing, impact, bouncing"

def build_graph(start_image, end_image, filename_prefix, positive_prompt):
    return {
        "checkpoint_loader": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": CKPT}},
        "positive": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["checkpoint_loader", 1], "text": positive_prompt}},
        "negative": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["checkpoint_loader", 1], "text": NEGATIVE_PROMPT}},
        "load_start": {"class_type": "LoadImage", "inputs": {"image": start_image}},
        "load_end": {"class_type": "LoadImage", "inputs": {"image": end_image}},
        # First-frame conditioning -- seeds the latent with the start image.
        "img2vid": {
            "class_type": "LTXVImgToVideo",
            "inputs": {
                "positive": ["positive", 0], "negative": ["negative", 0], "vae": ["checkpoint_loader", 2],
                "image": ["load_start", 0], "width": WIDTH, "height": HEIGHT, "length": LENGTH,
                "batch_size": 1, "strength": 1.0,
            },
        },
        # Adds the end image as a second guide frame at the last position -- this is
        # what turns plain image-to-video into first-LAST-frame conditioning.
        "add_end_guide": {
            "class_type": "LTXVAddGuide",
            "inputs": {
                "positive": ["img2vid", 0], "negative": ["img2vid", 1], "vae": ["checkpoint_loader", 2],
                "latent": ["img2vid", 2], "image": ["load_end", 0], "frame_idx": -1, "strength": 1.0,
            },
        },
        "conditioning": {
            "class_type": "LTXVConditioning",
            "inputs": {"positive": ["add_end_guide", 0], "negative": ["add_end_guide", 1], "frame_rate": FPS},
        },
        "sampler_select": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}},
        "scheduler": {
            "class_type": "LTXVScheduler",
            "inputs": {
                "steps": STEPS, "max_shift": 2.05, "base_shift": 0.95, "stretch": True,
                "terminal": 0.1, "latent": ["add_end_guide", 2],
            },
        },
        "sampler_custom": {
            "class_type": "SamplerCustom",
            "inputs": {
                "model": ["checkpoint_loader", 0], "add_noise": True, "noise_seed": SEED, "cfg": CFG,
                "positive": ["conditioning", 0], "negative": ["conditioning", 1],
                "sampler": ["sampler_select", 0], "sigmas": ["scheduler", 0],
                "latent_image": ["add_end_guide", 2],
            },
        },
        # Tiled decode -- avoids the VRAM-spike crash we hit with plain VAEDecode on Wan.
        "vae_decode": {
            "class_type": "VAEDecodeTiled",
            "inputs": {
                "samples": ["sampler_custom", 0], "vae": ["checkpoint_loader", 2],
                "tile_size": 256, "overlap": 64, "temporal_size": 32, "temporal_overlap": 8,
            },
        },
        "save_video": {
            "class_type": "SaveWEBM",
            "inputs": {"images": ["vae_decode", 0], "filename_prefix": filename_prefix, "codec": "vp9", "fps": float(FPS), "crf": 20.0},
        },
    }

def queue_prompt(graph):
    data = json.dumps({"prompt": graph}).encode("utf-8")
    req = urllib.request.Request(f"http://127.0.0.1:8188/prompt", data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

# Same timeout-bug fix as the Wan notebook -- give real long runs enough room,
# but LTX at this size should be far faster than A14B ever was.
def wait_for_completion(prompt_id, poll_interval=10, timeout=32000):
    start = time.time()
    while time.time() - start < timeout:
        with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as resp:
            hist = json.loads(resp.read())
        if prompt_id in hist:
            entry = hist[prompt_id]
            status = entry.get("status", {})
            if status.get("completed"):
                return entry
            if status.get("status_str") == "error":
                raise RuntimeError(f"Prompt {prompt_id} failed: {json.dumps(status, indent=2)}")
        elapsed = int(time.time() - start)
        if elapsed % 60 < poll_interval:
            print(f"  ...still running ({elapsed}s elapsed)")
        time.sleep(poll_interval)
    raise TimeoutError(f"Prompt {prompt_id} did not complete within {timeout}s")

def run_segment(name, start_image, end_image, filename_prefix, positive_prompt):
    print(f"=== Queuing segment: {name} ===")
    graph = build_graph(start_image, end_image, filename_prefix, positive_prompt)
    result = queue_prompt(graph)
    prompt_id = result["prompt_id"]
    print(f"  prompt_id={prompt_id}")
    entry = wait_for_completion(prompt_id)
    outputs = entry.get("outputs", {})
    video_info = outputs.get("save_video", {})
    print(f"  DONE: {json.dumps(video_info)}")
    return video_info

print(f'Ready ({"SMOKE TEST" if SMOKE_TEST else "FULL TEST"} mode). Run the next cell to generate.')


In [ ]:
result = run_segment("no-fence -> complete fence (LTX-Video test)", "start_no_fence.png", "end_full_fence.png", "fence_ltx", POSITIVE_PROMPT)
print("DONE")


In [ ]:
# Convert to mp4 and copy ONLY the small final file into /kaggle/working
# (the 20GB-capped, persisted directory) -- everything else stays in
# /kaggle/tmp and is discarded automatically when the session ends.
%cd /kaggle/tmp/ComfyUI/output
!ls -la *.webm

webm_file = [f for f in __import__('os').listdir('.') if f.startswith('fence_ltx')][0]
!ffmpeg -y -i "{webm_file}" -c:v libx264 -pix_fmt yuv420p -crf 18 /kaggle/working/fence_ltx.mp4
print('Final video at /kaggle/working/fence_ltx.mp4 -- check the Output tab after commit.')
